# پروژه‌ی اول بوت‌کمپ هوش مصنوعی کوئرا — بخش سوم، مسئله‌ی ۲ (الف): پیش‌بینی مبلغ اجاره

**تیم:** امین رفیع · مهدی طراح · میرامیررضا سیدنصیری · سعید حیاتی · مدل اجاره: امین رفیع

<div dir="rtl">

این نوت‌بوک نسخه‌ی تمیز و قابل‌اجرای مسیر نهایی مدل اجاره‌ی امین (`q4-5/ml_prediction-1.ipynb`) است.
نوت‌بوک اصلی ۱۷۵ سلول آزمایش داشت که بعضی‌هایشان بعد از ویرایش دیگر اجرا نمی‌شدند. این‌جا فقط تصمیم‌های نهایی، به همان ترتیب و با همان پارامترها، آمده است.

**نتایج اجرای اصلی** (روی `log1p(rent_value)`):

| مدل | Validation R² |
|---|---|
| Random Forest (مرحله‌ی اولیه) | 0.246 |
| Decision Tree | 0.424 |
| XGBoost پایه | 0.498 |
| XGBoost تنظیم‌شده | 0.558 |
| **XGBoost + مهندسی ویژگی (مدل نهایی)** | **0.612** |

روی داده‌ی Test: **R² = 0.602**، MAE = 0.685 و MSE = 1.240 (هر دو روی مقیاس لگاریتمی). فاصله‌ی Validation و Test حدود ۰٫۰۱ است، یعنی مدل روی داده‌ی Validation بیش‌برازش نکرده است.

**قواعد ارزیابی:** داده با `random_state=42` به ۸۰/۱۰/۱۰ تقسیم شده است. انتخاب ویژگی و پارامترها فقط با Validation انجام شده و Test فقط یک بار، برای مدل نهایی، استفاده شده است.
همه‌ی آمارهای گروهی (فراوانی محله، میانه‌ی متراژ و رهن در محله و شهر) فقط از Train ساخته می‌شوند تا نشت داده رخ ندهد.

</div>

## ۱. بارگذاری داده

In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.sparse import csr_matrix, hstack
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import OneHotEncoder
from sklearn.metrics import r2_score, mean_absolute_error, mean_squared_error
from xgboost import XGBRegressor

DIVAR_PATH = Path("../data/Divar.csv")
RANDOM_STATE = 42

COLUMNS = ["cat2_slug", "cat3_slug", "city_slug", "neighborhood_slug", "rent_value", "credit_value",
           "building_size", "rooms_count", "construction_year",
           "has_elevator", "has_parking", "has_warehouse", "has_heating_system"]
df = pd.read_csv(DIVAR_PATH, usecols=COLUMNS, low_memory=False)
print(df.shape)

## ۲. انتخاب آگهی‌های اجاره و پاک‌سازی

<div dir="rtl">

- فقط اجاره‌ی **مسکونی و تجاری** (اجاره‌ی روزانه/موقت جدا است و منطق قیمتش فرق دارد).
- ردیف‌هایی که رهن یا اجاره ندارند و مقادیر ۱۰۰ میلیارد به بالا (اشتباه تایپی) حذف شدند.
- **رهن کل معادل** (`total_credit` = رهن + اجاره × ۱۰۰/۳) فقط برای حذف آگهی‌های بی‌معنی استفاده شد: بیش از ۵۰۰ میلیارد یا کمتر از یک میلیون.
- متراژ غیرممکن **برای هر نوع ملک جداگانه** حذف شد (مثلاً آپارتمان بالای ۵۰۰ متر). متراژهای بزرگ ولی ممکن نگه داشته شدند، چون IQR ساده املاک بزرگ واقعی را هم حذف می‌کرد.

</div>

In [ ]:
rental = df[df["cat2_slug"].isin(["residential-rent", "commercial-rent"])].copy()
print("rental ads:", len(rental))

rental = rental[rental["rent_value"].notna() & rental["credit_value"].notna()
                & (rental["rent_value"] < 1e11) & (rental["credit_value"] < 1e11)]
rental["total_credit"] = rental["credit_value"] + rental["rent_value"] * (100_000_000 / 3_000_000)
rental = rental[rental["total_credit"].between(1e6, 5e11)]

SIZE_LIMITS = {"apartment-rent": 500, "house-villa-rent": 2_000, "shop-rent": 2_000,
               "office-rent": 2_000, "industry-agriculture-business-rent": 500_000}
too_big = rental["building_size"] > rental["cat3_slug"].map(SIZE_LIMITS).fillna(np.inf)
rental = rental[~too_big]
print("after cleaning:", len(rental))

## ۳. متغیر هدف

<div dir="rtl">

- حدود ۱۷٪ آگهی‌ها اجاره‌ی **صفر** دارند. صفر یعنی «رهن کامل» یا «توافقی» و مبلغ واقعی اجاره نیست، پس از مجموعه‌ی هدف حذف شدند.
- `rent_value` به‌شدت راست‌چوله است (میانه ۶٫۳ میلیون، بیشینه ۱۳ میلیارد). مدل روی `log1p(rent_value)` آموزش می‌بیند تا چند اجاره‌ی خیلی بزرگ یادگیری را در دست نگیرند.

</div>

In [ ]:
print("zero rent share:", round((rental["rent_value"] == 0).mean(), 4))
rent = rental[rental["rent_value"] > 0].copy()
print("records used for modelling:", len(rent))
print((rent["rent_value"].describe(percentiles=[0.01, 0.25, 0.5, 0.75, 0.99]) / 1e6).round(2).rename("rent (million Toman)"))

fig, axes = plt.subplots(1, 2, figsize=(14, 4))
axes[0].hist(rent["rent_value"] / 1e6, bins=100)
axes[0].set_title("rent_value (million Toman)")
axes[1].hist(np.log1p(rent["rent_value"]), bins=100, color="#27ae60")
axes[1].set_title("log1p(rent_value)")
plt.tight_layout()
plt.show()

## ۴. ویژگی‌های پایه

In [ ]:
FA_DIGITS = str.maketrans("۰۱۲۳۴۵۶۷۸۹", "0123456789")
year = rent["construction_year"].astype("string").str.translate(FA_DIGITS).str.replace("قبل از 1370", "1369", regex=False)
rent["building_age"] = 1405 - pd.to_numeric(year, errors="coerce")

ROOMS = {"بدون اتاق": 0, "یک": 1, "دو": 2, "سه": 3, "چهار": 4, "پنج یا بیشتر": 5}
rent["rooms_count_numeric"] = rent["rooms_count"].map(ROOMS)

TRUE_FALSE = {True: 1, "true": 1, "True": 1, False: 0, "false": 0, "False": 0}
for column in ["has_elevator", "has_parking", "has_warehouse"]:
    rent[column] = rent[column].map(TRUE_FALSE).fillna(0)  # not mentioned in the ad -> treated as absent

for column in ["has_heating_system", "cat3_slug", "city_slug", "neighborhood_slug"]:
    rent[column] = rent[column].fillna("Unknown").astype(str)

## ۵. تقسیم Train / Validation / Test

In [ ]:
train, temp = train_test_split(rent, test_size=0.20, random_state=RANDOM_STATE)
validation, test = train_test_split(temp, test_size=0.50, random_state=RANDOM_STATE)
train, validation, test = train.copy(), validation.copy(), test.copy()
print("train:", len(train), " validation:", len(validation), " test:", len(test))

# missing age / rooms are filled with the TRAIN median only
for column in ["building_age", "rooms_count_numeric"]:
    median = train[column].median()
    for part in (train, validation, test):
        part[column] = part[column].fillna(median)

## ۶. مهندسی ویژگی (آمارها فقط از Train)

<div dir="rtl">

| ویژگی | معنی |
|---|---|
| `credit_value_per_size` | رهن به ازای هر متر، سطح قیمت ملک مستقل از اندازه |
| `neighborhood_frequency` | تعداد آگهی‌های محله در Train، نشانه‌ی پرتقاضا بودن محله |
| `neighborhood_share_in_city` | سهم محله از آگهی‌های شهر؛ به‌تنهایی حدود ۰٫۰۰۹ به R² اضافه کرد |
| `building_size_neighborhood_ratio` | متراژ نسبت به متراژ معمول محله |
| `credit_value_neighborhood_median_ratio` | رهن نسبت به رهن معمول محله |
| `credit_value_city_median_ratio` | رهن نسبت به رهن معمول شهر |

مجموع این ویژگی‌ها Validation R² را از ۰٫۵۹۱ به ۰٫۶۱۲ رساند. ویژگی‌هایی مثل `city_type`، `luxury_score` و `has_cooling_system` هم آزمایش شدند ولی بهبودی ندادند و کنار گذاشته شدند.

</div>

In [ ]:
def add_engineered_features(part, reference):
    """Add ratio / frequency features; every group statistic comes from `reference` (the train split)."""
    part = part.copy()
    size = part["building_size"].replace(0, np.nan)
    part["credit_value_per_size"] = part["credit_value"] / size

    neighborhood_count = reference["neighborhood_slug"].value_counts()
    city_count = reference["city_slug"].value_counts()
    part["neighborhood_frequency"] = part["neighborhood_slug"].map(neighborhood_count)
    part["neighborhood_share_in_city"] = part["neighborhood_frequency"] / part["city_slug"].map(city_count)

    size_by_neighborhood = reference.groupby("neighborhood_slug")["building_size"].median()
    credit_by_neighborhood = reference.groupby("neighborhood_slug")["credit_value"].median()
    credit_by_city = reference.groupby("city_slug")["credit_value"].median()
    part["building_size_neighborhood_ratio"] = size / part["neighborhood_slug"].map(size_by_neighborhood).replace(0, np.nan)
    part["credit_value_neighborhood_median_ratio"] = part["credit_value"] / part["neighborhood_slug"].map(credit_by_neighborhood).replace(0, np.nan)
    part["credit_value_city_median_ratio"] = part["credit_value"] / part["city_slug"].map(credit_by_city).replace(0, np.nan)
    return part.replace([np.inf, -np.inf], np.nan)


reference = train.copy()
train = add_engineered_features(train, reference)
validation = add_engineered_features(validation, reference)
test = add_engineered_features(test, reference)

## ۷. ماتریس ویژگی‌ها

In [ ]:
CATEGORICAL = ["city_slug", "cat2_slug", "neighborhood_slug", "has_heating_system", "cat3_slug"]
BASE_NUMERIC = ["building_size", "building_age", "rooms_count_numeric", "has_elevator", "has_parking", "has_warehouse"]
CREDIT_NUMERIC = ["credit_value", "credit_value_per_size", "credit_value_neighborhood_median_ratio", "credit_value_city_median_ratio"]
ENGINEERED = ["neighborhood_frequency", "neighborhood_share_in_city", "building_size_neighborhood_ratio"]


def build_matrices(numeric):
    encoder = OneHotEncoder(handle_unknown="ignore")
    encoder.fit(train[CATEGORICAL])
    matrices = [hstack([csr_matrix(part[numeric].to_numpy(dtype=float)), encoder.transform(part[CATEGORICAL])]).tocsr()
                for part in (train, validation, test)]
    names = numeric + list(encoder.get_feature_names_out(CATEGORICAL))
    return matrices, names


NUMERIC = CREDIT_NUMERIC + BASE_NUMERIC + ENGINEERED
(X_train, X_validation, X_test), feature_names = build_matrices(NUMERIC)
y_train, y_validation, y_test = (np.log1p(part["rent_value"]) for part in (train, validation, test))
print(X_train.shape, X_validation.shape, X_test.shape)

## ۸. مدل نهایی: XGBoost

<div dir="rtl">

پارامترها به‌صورت مرحله‌ای و فقط با Validation تنظیم شدند: اول `max_depth` و `learning_rate`، بعد `n_estimators` و `min_child_weight`، و در آخر `gamma`، `subsample` و `colsample_bytree`.
XGBoost از Random Forest و Decision Tree بهتر بود: بوستینگ خطای مدل‌های قبلی را مرحله‌به‌مرحله اصلاح می‌کند و با هزاران ستون تُنُک one-hot (محله‌ها) هم به‌خوبی کار می‌کند.

⏱ آموزش ۳۰۰۰ درخت با عمق ۱۰ روی حدود ۲۳۰ هزار ردیف چند دقیقه طول می‌کشد.

</div>

In [ ]:
XGB_PARAMS = dict(n_estimators=3000, max_depth=10, min_child_weight=5, learning_rate=0.05, gamma=0,
                  subsample=0.8, colsample_bytree=0.8, objective="reg:squarederror",
                  tree_method="hist", random_state=RANDOM_STATE, n_jobs=-1)

model = XGBRegressor(**XGB_PARAMS)
model.fit(X_train, y_train)

## ۹. ارزیابی

In [ ]:
def report(y_true_log, y_pred_log, name):
    """Metrics on the log scale (what the model optimises) and in Toman (what a user sees)."""
    true_toman, pred_toman = np.expm1(y_true_log), np.expm1(y_pred_log)
    return {
        "split": name,
        "R2 (log)": r2_score(y_true_log, y_pred_log),
        "MAE (log)": mean_absolute_error(y_true_log, y_pred_log),
        "MSE (log)": mean_squared_error(y_true_log, y_pred_log),
        "R2 (Toman)": r2_score(true_toman, pred_toman),
        "MAE (million Toman)": mean_absolute_error(true_toman, pred_toman) / 1e6,
        "MSE (Toman^2)": mean_squared_error(true_toman, pred_toman),
        "median abs % error": np.median(np.abs(pred_toman - true_toman) / true_toman) * 100,
    }


validation_pred = model.predict(X_validation)
results = pd.DataFrame([report(y_train, model.predict(X_train), "train"),
                        report(y_validation, validation_pred, "validation")]).set_index("split")
results.round(4)

<div dir="rtl">

Test فقط **یک بار** و بعد از قطعی شدن ویژگی‌ها و پارامترها اجرا می‌شود:

</div>

In [ ]:
test_pred = model.predict(X_test)
results.loc["test"] = pd.Series(report(y_test, test_pred, "test")).drop("split")
results.round(4)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(15, 6))

axes[0].hexbin(y_test, test_pred, gridsize=80, bins="log", cmap="viridis")
lims = [min(y_test.min(), test_pred.min()), max(y_test.max(), test_pred.max())]
axes[0].plot(lims, lims, color="red", linewidth=1)
axes[0].set_xlabel("actual log1p(rent)")
axes[0].set_ylabel("predicted log1p(rent)")
axes[0].set_title("Test set: predicted vs actual")

buckets = pd.cut(np.expm1(y_test), [0, 2e6, 5e6, 1e7, 2e7, 5e7, 1e8, np.inf],
                 labels=["<2M", "2-5M", "5-10M", "10-20M", "20-50M", "50-100M", ">100M"])
error = pd.DataFrame({"bucket": buckets, "abs_pct_error": np.abs(np.expm1(test_pred) - np.expm1(y_test)) / np.expm1(y_test) * 100})
error.groupby("bucket", observed=True)["abs_pct_error"].median().plot(kind="bar", ax=axes[1], color="#e67e22")
axes[1].set_ylabel("median absolute % error")
axes[1].set_xlabel("actual rent")
axes[1].set_title("Test error by rent range")
plt.tight_layout()
plt.show()

In [ ]:
importance = pd.Series(model.feature_importances_, index=feature_names)
grouped = importance.groupby(lambda name: next((c for c in CATEGORICAL if name.startswith(c + "_")), name)).sum()
grouped.sort_values().tail(15).plot(kind="barh", figsize=(9, 6), title="Feature importance (one-hot columns summed per feature)")
plt.tight_layout()
plt.show()

## ۱۰. بدون رهن: مدل بدون `credit_value`

<div dir="rtl">

مهم‌ترین ورودی مدل، **رهن همان آگهی** است. رهن و اجاره دو بخش یک قیمت‌اند و با هم تعیین می‌شوند، پس مدل بالا در واقع به این سؤال جواب می‌دهد:
«با این مشخصات و **این مقدار رهن**، اجاره چقدر است؟». این نشت متغیر هدف نیست، چون رهن پیش از اجاره در آگهی معلوم است، ولی بخش بزرگی از دقت مدل از همین ستون می‌آید.

برای سنجیدن سهم ویژگی‌های خود ملک، همان مدل بدون هیچ ویژگی وابسته به رهن آموزش داده می‌شود. ⏱ این سلول اختیاری است و آموزش آن هم چند دقیقه طول می‌کشد.

</div>

In [ ]:
RUN_WITHOUT_CREDIT = True

if RUN_WITHOUT_CREDIT:
    (Xn_train, Xn_validation, Xn_test), _ = build_matrices(BASE_NUMERIC + ENGINEERED)
    model_no_credit = XGBRegressor(**XGB_PARAMS).fit(Xn_train, y_train)
    comparison = pd.DataFrame([
        report(y_validation, validation_pred, "with credit (validation)"),
        report(y_validation, model_no_credit.predict(Xn_validation), "without credit (validation)"),
    ]).set_index("split")
    display(comparison.round(4))

## ۱۱. جمع‌بندی

<div dir="rtl">

- مدل نهایی، XGBoost روی `log1p(rent_value)` با ویژگی‌های مکانی و نسبتی، در اجرای اصلی به **R² برابر ۰٫۶۰۲ روی Test** رسید و فاصله‌اش با Validation حدود ۰٫۰۱ بود. یعنی تعمیم مدل پایدار است.
- MAE و MSE روی مقیاس لگاریتمی گزارش شده‌اند. معادل تومانی آن‌ها در جدول ارزیابی آمده؛ R² تومانی کمتر است، چون چند اجاره‌ی خیلی بزرگ در MSE تومانی وزن زیادی دارند.
- خطا در اجاره‌های بالای ۱۰۰ میلیون بیشتر است و مدل آن‌ها را **کم‌برآورد** می‌کند. نمونه‌ی این بازه کم است و تعدادی از آن‌ها احتمالاً اشتباه تایپی‌اند (مثل ۱۱٬۱۱۱٬۱۱۱٬۱۱۱).
- مهم‌ترین ویژگی‌ها: رهن، متراژ، شهر و محله. ویژگی‌هایی که ملک را با محله و شهرش مقایسه می‌کنند بیشترین سود را از مهندسی ویژگی دادند.
- مقایسه‌ی بخش ۱۰ نشان می‌دهد چقدر از دقت به دانستن رهن وابسته است. در ارائه حتماً به این نکته اشاره کنید.

</div>